In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.axes_grid1 import inset_locator
import matplotlib as mpl
from numba import njit, prange

mpl.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 18,
    'text.usetex': False  # Set to True if your local machine has a LaTeX installation
})

@njit(parallel=True)
def reconstruct_physics_numba(pos, radius, k_spring, sample_fraction=0.2):
    """
    Blazing fast parallel structural reconstruction.
    Samples distances to keep memory low on large N datasets.
    """
    N = len(pos)
    cutoff = 2.0 * radius
    
    # Estimate contact lines (pre-allocate maximum safe array size)
    max_lines = N * 10
    lines_out = np.zeros((max_lines, 2, 2), dtype=np.float64)
    force_mags = np.zeros(max_lines, dtype=np.float64)
    
    # Thread-safe accumulation requires a loop over particles
    line_counts = np.zeros(N, dtype=np.int64)
    
    # 1. Extract Force Chains
    for i in prange(N):
        idx_start = i * 10
        count = 0
        for j in range(N):
            if i == j: continue
            dx_raw = pos[i, 0] - pos[j, 0]
            dy_raw = pos[i, 1] - pos[j, 1]
            dx = dx_raw - np.round(dx_raw)
            dy = dy_raw - np.round(dy_raw)
            dist = np.sqrt(dx**2 + dy**2)
            
            if dist < cutoff and j > i: # Avoid double counting lines
                if count < 10:
                    global_idx = idx_start + count
                    lines_out[global_idx, 0, 0] = pos[i, 0]
                    lines_out[global_idx, 0, 1] = pos[i, 1]
                    lines_out[global_idx, 1, 0] = pos[i, 0] - dx
                    lines_out[global_idx, 1, 1] = pos[i, 1] - dy
                    force_mags[global_idx] = k_spring * (cutoff - dist)
                    count += 1
        line_counts[i] = count

    # Clean and consolidate line structures
    valid_lines = []
    valid_forces = []
    for i in range(N):
        for c in range(line_counts[i]):
            g_idx = i * 10 + c
            valid_lines.append(lines_out[g_idx])
            valid_forces.append(force_mags[g_idx])
            
    # 2. Sample Distances for g(r) to prevent memory crash on laptop
    sample_size = int(N * sample_fraction)
    sampled_indices = np.random.choice(N, sample_size, replace=False)
    distances_out = np.zeros((sample_size * sample_size), dtype=np.float64)
    
    idx = 0
    for i in range(sample_size):
        ii = sampled_indices[i]
        for j in range(sample_size):
            jj = sampled_indices[j]
            if ii == jj: continue
            dx_raw = pos[ii, 0] - pos[jj, 0]
            dy_raw = pos[ii, 1] - pos[jj, 1]
            dx = dx_raw - np.round(dx_raw)
            dy = dy_raw - np.round(dy_raw)
            distances_out[idx] = np.sqrt(dx**2 + dy**2)
            idx += 1
            
    return valid_lines, np.array(valid_forces), distances_out[:idx]

# Change this path to point to your actual downloaded dataset
filename = "results/PRL_Dataset_N5000_Grid8.npz" 

print(f"Loading HPC Data from {filename}...")
data = np.load(filename)

pos_async = data['pos_async']
pos_glob = data['pos_glob']
dt_grid_snap = data['dt_grid_snap']
e_hist_async = data['e_hist_async']
e_hist_glob = data['e_hist_glob']
dt_hist_async = data['dt_hist_async']
dt_hist_glob = data['dt_hist_glob']
time_async = float(data['time_async'])
time_glob = float(data['time_glob'])
accumulated_dt = data['accumulated_dt']
RADIUS = float(data['radius'])
K_SPRING = float(data['k_spring'])
GRID_DIVS = int(data['grid_divs'])
N_ATOMS = int(data['n_atoms'])

print(f"Data Loaded Successfully!")
print(f"N_ATOMS: {N_ATOMS} | GRID: {GRID_DIVS}x{GRID_DIVS}")
print(f"Global Wall-Clock Time: {time_glob:.2f}s | Async Wall-Clock Time: {time_async:.2f}s")

# Run Numba structural optimization helper
print("\nExecuting optimized parallel structural analysis...")
lines, force_array, distances = reconstruct_physics_numba(pos_async, RADIUS, K_SPRING)
print(f"Structural analysis complete. Reconstructed {len(lines)} active contact networks.")


# ------------------------------------------
# FIGURE 1: Phase Space Spatial Mapping
# ------------------------------------------
print("Generating Figure 1: Spatial Time-Step Heatmap...")
fig1, ax1 = plt.subplots(figsize=(8, 7))
dt_grid = dt_grid_snap.reshape(GRID_DIVS, GRID_DIVS).T
heatmap1 = ax1.imshow(dt_grid, cmap='Blues_r', origin='lower', extent=[0, 1, 0, 1], alpha=0.65)
ax1.scatter(pos_async[:, 0], pos_async[:, 1], s=0.7, c='black', alpha=0.15)

if len(lines) > 0:
    lw = (force_array / np.max(force_array)) * 2.2 + 0.3
    lc = LineCollection(lines, cmap='autumn', linewidths=lw)
    lc.set_array(force_array)
    ax1.add_collection(lc)

ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_xticks([]); ax1.set_yticks([])
fig1.colorbar(heatmap1, ax=ax1, fraction=0.046, pad=0.04, label="Local Domain Time Step ($\Delta t_k$)")
ax1.set_title("Autonomous Force-Chain Tracking Landscape", pad=10)
plt.savefig('Fig1_Bifurcation.pdf', bbox_inches='tight')
plt.close()

# ------------------------------------------
# FIGURE 2: Dynamical Bifurcation & Histograms
# ------------------------------------------
print("Generating Figure 2: Algorithmic Timestep Dynamics...")
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Step Histogram
ax2a.hist(dt_grid_snap, bins=15, color='rebeccapurple', edgecolor='black', alpha=0.85)
ax2a.set_yscale('log')
ax2a.set_xlabel("Local Domain Time Step Size")
ax2a.set_ylabel("Domain Bin Count (Log Scale)")
ax2a.set_title("Bimodal Domain Timestep Phase Separation")
ax2a.grid(alpha=0.25)

# Panel B: Time Step Tracks
steps_glob = np.arange(len(dt_hist_glob)) * 10
steps_async = np.arange(len(dt_hist_async)) * 10
ax2b.plot(steps_glob, dt_hist_glob, 'k--', linewidth=1.5, label='Global FIRE Baseline')
ax2b.plot(steps_async, dt_hist_async, 'r-', linewidth=2, label='Async FIRE (Domain Average)')
ax2b.set_xlabel("Algorithmic Integration Steps")
ax2b.set_ylabel("Time Step Size ($\Delta t$)")
ax2b.set_title("Global Synchronous Collapse vs. Decoupled Velocity")
ax2b.legend(loc='lower right')
ax2b.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('Fig2_Dynamics.pdf', bbox_inches='tight')
plt.close()

# ------------------------------------------
# FIGURE 3: Topological Invariance & g(r)
# ------------------------------------------
print("Generating Figure 3: Algorithmic Step Energy Minimization...")
fig3, ax3 = plt.subplots(figsize=(8, 6))
steps_e_glob = np.arange(len(e_hist_glob)) * 10
steps_e_async = np.arange(len(e_hist_async)) * 10

ax3.plot(steps_e_glob, e_hist_glob, 'k--', linewidth=2, label='Global FIRE Baseline')
ax3.plot(steps_e_async, e_hist_async, 'r-', linewidth=2, label='Async FIRE Engine')
ax3.set_yscale('log')
ax3.set_xlabel("Algorithmic Integration Steps")
ax3.set_ylabel("Total System Potential Energy ($U$)")
ax3.set_title("Structural Energy Relaxation Invariance")
ax3.legend(loc='lower left')
ax3.grid(alpha=0.25)

# Inset plot for Radial Distribution Function g(r)
axins = inset_locator.inset_axes(ax3, width="42%", height="42%", loc='upper right')
dr = 0.002
r_max = 4.5 * RADIUS
bins = np.arange(0, r_max, dr)
hist, _ = np.histogram(distances, bins=bins)
r_centers = (bins[1:] + bins[:-1]) / 2.0

# Normalize g(r) based on sampled density
sample_density = len(distances) / (np.pi * (r_max**2))
ideal_counts = 2.0 * np.pi * r_centers * dr * (len(pos_async) / (1.0**2)) * (len(distances)/len(pos_async))
g_r = hist / ideal_counts

axins.plot(r_centers / (2.0*RADIUS), g_r, color='royalblue', linewidth=1.5)
axins.axvline(1.0, color='crimson', linestyle=':', alpha=0.8, label='$2R$')
axins.set_xlim(0.8, 2.2)
axins.set_xlabel("$r / 2R$", fontsize=9)
axins.set_ylabel("$g(r)$", fontsize=9)
axins.set_title("Isostatic Pair Distribution", fontsize=9)
plt.savefig('Fig3_Energy_GR.pdf', bbox_inches='tight')
plt.close()

# ------------------------------------------
# FIGURE 4: HPC Wall-Clock Speedup Summary
# ------------------------------------------
print("Generating Figure 4: Total Time-To-Solution...")
fig4, ax4 = plt.subplots(figsize=(6, 5))
labels = ['Global FIRE', 'Async FIRE']
times = [time_glob, time_async]
colors = ['#333333', 'crimson']

bars = ax4.bar(labels, times, color=colors, width=0.45, edgecolor='black', zorder=3)
ax4.set_ylabel("Total Wall-Clock Optimization Time (Seconds)")
ax4.set_title(f"HPC Node Performance Metric (N={N_ATOMS})")
ax4.grid(axis='y', linestyle=':', alpha=0.5, zorder=0)

speedup = time_glob / time_async
ax4.text(1, time_async + (time_glob * 0.03), f"{speedup:.2f}x Total Speedup", 
         ha='center', va='bottom', fontweight='bold', color='crimson', fontsize=11)
plt.savefig('Fig4_Speedup.pdf', bbox_inches='tight')
plt.close()

# ------------------------------------------
# FIGURE 5: Accumulated Virtual Age Heatmap
# ------------------------------------------
print("Generating Figure 5: Accumulated Phase-Space Age...")
fig5, ax5 = plt.subplots(figsize=(8, 7))
age_grid = accumulated_dt.reshape(GRID_DIVS, GRID_DIVS).T
heatmap5 = ax5.imshow(age_grid, cmap='magma', origin='lower', extent=[0, 1, 0, 1], alpha=0.85)

if len(lines) > 0:
    lw = (force_array / np.max(force_array)) * 1.4 + 0.1
    lc5 = LineCollection(lines, colors='white', linewidths=lw, alpha=0.35)
    ax5.add_collection(lc5)

ax5.set_xlim(0, 1); ax5.set_ylim(0, 1)
ax5.set_xticks([]); ax5.set_yticks([])
fig5.colorbar(heatmap5, ax=ax5, fraction=0.046, pad=0.04, label=r"Accumulated Local Virtual Timeline ($\sum \Delta t_k$)")
ax5.set_title("Phase Space 'Virtual Age' Discrepancy Map", pad=10)
plt.savefig('Fig5_Virtual_Age.pdf', bbox_inches='tight')
plt.close()

# ------------------------------------------
# FIGURE 6: Energy Decay vs. Continuous Time
# ------------------------------------------
print("Generating Figure 6: Continuous Wall-Clock Energy Trajectory...")
fig6, ax6 = plt.subplots(figsize=(8, 6))

time_arr_glob = np.linspace(0, time_glob, len(e_hist_glob))
time_arr_async = np.linspace(0, time_async, len(e_hist_async))

ax6.plot(time_arr_glob, e_hist_glob, 'k--', linewidth=2, label='Global FIRE Baseline')
ax6.plot(time_arr_async, e_hist_async, 'r-', linewidth=2, label='Async FIRE Engine')
ax6.set_yscale('log')
ax6.set_xlabel("Elapsed Continuous Compute Time (Seconds)")
ax6.set_ylabel("Total System Potential Energy ($U$)")
ax6.set_title("Optimization Rate vs. Absolute Processing Time")
ax6.legend(loc='lower left')
ax6.grid(alpha=0.25)
plt.savefig('Fig6_Energy_vs_Time.pdf', bbox_inches='tight')
plt.close()

print("\nSuccess! All 6 high-resolution publication figures written to individual PDF vector files.")